# 00 - Quickstart: environment and first run

This quickstart notebook runs a compact end-to-end parametric workflow so you can validate the environment and execution path before larger studies.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `get_building`
- `accis_single_idf_funcs.reduce_runtime`
- `ParametricSimulation`
- `ParametricSimulation.run_parametric_simulation`
- `WorkspaceArtifactCleaner`
- `ef.get_building`
- `parametric.discover_available_outputs`
- `parametric.select_outputs`
- `parametric.apply_outputs_preflight`
- `parametric.set_outputs_for_simulation`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## Quickstart run: import workflow dependencies

We start by importing the APIs required for this notebook block.

Keeping imports explicit at the top of each topic notebook makes selective reruns safer when you jump between sections of the series.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
from pathlib import Path

import accim
import pandas as pd
from besos import eppy_funcs as ef
from accim.utils import WorkspaceArtifactCleaner
from accim.parametric_and_optimisation.main import ParametricSimulation

pd.set_option("display.max_columns", 120)


## Quickstart run: define paths and execution folders

This block defines supporting paths and run constants used by later cells.

Centralizing these values reduces hidden coupling and keeps the execution flow easier to audit.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `idf_file`: reused in subsequent workflow blocks.
- `epw_list`: reused in subsequent workflow blocks.
- `RESULTS_DIRNAME`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `PROJECT_ROOT`, `idf_file`, `epw_list`, `RESULTS_DIRNAME`.

In [ ]:
PROJECT_ROOT = Path.cwd()

idf_file = "OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance.idf"
epw_list = ["Seville.epw"]
RESULTS_DIRNAME = "nb00_quickstart_results"

print("Project root:", PROJECT_ROOT)
print("IDF exists:", (PROJECT_ROOT / idf_file).exists())
print("EPW exists:", (PROJECT_ROOT / epw_list[0]).exists())
print("Note: quickstart uses the high-performance envelope IDF only.")


## Quickstart run: capture baseline workspace state for cleanup

We initialize artifact tracking so generated files can be reviewed and cleaned in a controlled way.

This is especially useful in iterative notebook sessions where repeated runs can quickly accumulate temporary folders and CSV files.

Key arguments used in this cell and why they matter:
- `workspace_root`: defines this setting for the current call.

Variables created in this cell:
- `artifact_cleaner`: reused in subsequent workflow blocks.
- `baseline_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `artifact_cleaner`, `baseline_files`.

In [ ]:
artifact_cleaner = WorkspaceArtifactCleaner(workspace_root=PROJECT_ROOT)
baseline_files = artifact_cleaner.capture_initial_state()
print("Baseline file count:", len(baseline_files))


## Quickstart run: load the baseline IDF model

Load the base building model used by subsequent runs.

This is where the baseline IDF is loaded for all downstream operations. Later cells reuse this object for output setup, parameterization, and execution.

Variables created in this cell:
- `building`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `building`.

In [ ]:
building = ef.get_building(idf_file)


## Quickstart run: apply runtime-reduction settings

Apply runtime reductions to keep notebook runs faster during exploration.

This block shortens exploratory runs by reducing simulation burden. Use it for prototyping and switch to fuller settings when producing final results.

Key arguments used in this cell and why they matter:
- `idf_object`: defines this setting for the current call.
- `timesteps`: defines this setting for the current call.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:

# Keep quickstart fast. Increase timesteps for production studies.
accim.utils.reduce_runtime(idf_object=building, timesteps=2)


## Quickstart run: initialize parametric simulation object

Create the parametric simulation object with weather context and model settings.

This creates the parametric simulation object that will orchestrate outputs, parameters, and run plans. From here on, the workflow uses this object as the main controller.

Key arguments used in this cell and why they matter:
- `building`: loaded IDF model used in this workflow.
- `parameters_type`: parameter family used in the simulation setup.
- `epws`: list of EPW weather files used to run scenarios.
- `output_freqs`: time frequency requested for outputs.

Variables created in this cell:
- `parametric`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `parametric`.

In [ ]:

parametric = ParametricSimulation(
	building=building,
	parameters_type="accim custom model",
	epws=epw_list,
	output_freqs=["hourly"],
)


## Quickstart run: discover available meter and variable outputs

Inspect available meters and variables before selecting valid outputs.

Before selecting outputs, we inspect what meters and variables are currently available. That discovery step reduces mistakes when requesting objective or analysis outputs.

Key arguments used in this cell and why they matter:
- `reduce_sim_time`: enables a fast scan mode when discovering outputs.
- `prefer`: preferred backend for output discovery.

Variables created in this cell:
- `df_meters_available`: reused in subsequent workflow blocks.
- `df_vars_available`: reused in subsequent workflow blocks.
- `outputs_meta`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check that the discovered meter/variable tables are not unexpectedly empty.
- Confirm the reported discovery source matches your preferred backend.

In [ ]:
df_meters_available, df_vars_available, outputs_meta = parametric.discover_available_outputs(
	reduce_sim_time=True,  # lightweight discovery helper run
	prefer="testsimeplus",  # fallback policy when multiple discovery backends exist
)

print("Discovery source:", outputs_meta.get("source"))
print("Meters available:", len(df_meters_available))
print("Variables available:", len(df_vars_available))


## Quickstart run: select objective and analysis outputs

Select target outputs and validate them before applying changes to the model.

This section applies explicit output selection and preflight validation. Validating before execution keeps result schemas stable across runs.

Key arguments used in this cell and why they matter:
- `meters`: energy meters to include as outputs.
- `match`: matching rule used to select outputs.
- `on_missing`: behavior when a requested output is missing.
- `suggest`: shows suggestions when no exact match is found.

Variables created in this cell:
- `objective_meters`: reused in subsequent workflow blocks.
- `df_meters_sel`: reused in subsequent workflow blocks.
- `_`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [ ]:
objective_meters = ["Heating:Electricity", "Cooling:Electricity"]
df_meters_sel, _, _ = parametric.select_outputs(
	meters=objective_meters,  # exact output goals for this quickstart
	match="case_insensitive",  # robust matching for common casing differences
	on_missing="raise",  # fail early if an objective output cannot be found
	suggest=True,  # print close matches when selection fails
)


## Quickstart run: run output preflight checks before execution

Select target outputs and validate them before applying changes to the model.

This section applies explicit output selection and preflight validation. Validating before execution keeps result schemas stable across runs.

Key arguments used in this cell and why they matter:
- `df_meters_sel`: table of selected meters to apply to the IDF.
- `df_vars_sel`: table of selected variables to apply to the IDF.
- `clean_mode`: cleanup mode applied before adding outputs.
- `validate_before_apply`: whether to validate before editing the IDF.
- `validate_after_apply`: whether to validate after editing the IDF.
- `on_missing`: behavior when a requested output is missing.

Variables created in this cell:
- `preflight`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [ ]:
preflight = parametric.apply_outputs_preflight(
	df_meters_sel=df_meters_sel,
	df_vars_sel=None,
	clean_mode="all",  # remove legacy output objects and keep selected ones
	validate_before_apply=False,
	validate_after_apply=True,  # confirm selected outputs exist after apply
	on_missing="raise",
)

print("Missing meters after apply:", len(preflight["verification"]["meters"]["missing_in_idf"]))


## Quickstart run: apply selected output schema to simulation settings

Apply the validated output meter/variable schema to the simulation object.

This block applies the validated output-meter and output-variable schema to the active simulation object. From this point on, execution calls will use this explicit output definition.

Key arguments used in this cell and why they matter:
- `df_output_meter`: final meter table used by the simulation.

What to verify right after running this cell:
- Confirm output schema assignment completed without warnings.
- Verify target meter/variable names match the intended analysis outputs.

In [ ]:
parametric.set_outputs_for_simulation(df_output_meter=df_meters_sel)


## Quickstart run: define decision variables and objective directions

Set parameter ranges or values and build the study problem.

Here we define the decision space and objective orientation used in simulation or optimization. These choices directly determine how the final result space will look.

Key arguments used in this cell and why they matter:
- `accis_params_dict`: ACCIS/aPMV parameter definition used in the study.
- `num_samples`: number of samples in the sampling plan.

Variables created in this cell:
- `accis_parameters`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `accis_parameters`.

In [ ]:
# Define a tiny parameter space and attach the EPProblem.
# Each tuple is interpreted as a numeric range: (min, max).
accis_parameters = {
	"CustAST_m": (0.2, 0.8),
	"CustAST_n": (10, 22),
	"CustAST_ASToffset": (2.0, 5.0),
}

parametric.set_parameters(accis_params_dict=accis_parameters)
parametric.set_problem()  # builds EPProblem from selected outputs + parameter definition
parametric.sampling_lhs(num_samples=4)  # small Latin Hypercube sample for demonstration

print("Sample size:", len(parametric.parameters_values_df))
parametric.parameters_values_df.head()


## Quickstart run: execute parametric run and inspect output rows

This is the execution checkpoint for the configured parametric plan.

After the run, verify row counts and core objective columns immediately so issues are caught before post-processing.

Key arguments used in this cell and why they matter:
- `out_dir`: output folder for generated artifacts and results.
- `df`: case plan dataframe executed in this run.
- `processes`: number of parallel processes for simulation runs.
- `keep_input`: whether generated input files are preserved.
- `keep_dirs`: whether per-run directories are preserved.

Variables created in this cell:
- `outputs`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare returned row count with the expected number of plan combinations.
- Inspect the first rows to ensure objective columns are present and populated.

In [ ]:
outputs = parametric.run_parametric_simulation(
	out_dir=RESULTS_DIRNAME,  # output folder for batch artifacts and logs
	df=parametric.parameters_values_df,  # explicit sample plan to execute
	processes=1,  # increase with care; parallel overhead can dominate tiny runs
	keep_input=True,  # keep generated input models for auditability
	keep_dirs=False,  # remove per-run directories to reduce disk usage
)
print("Rows:", len(outputs))
print(outputs.head())


### Reading the result

Review the row count and confirm it matches the expected combinations from your plan and EPW set.

## Quickstart run: preview execution outputs and row counts

Inspect a compact preview of execution outputs before proceeding.

This cell previews key output rows or summary counts after execution. Use it to quickly validate shape and columns before moving to deeper analysis.

What to verify right after running this cell:
- Confirm row counts and first records look consistent with the execution settings.
- Check that expected objective/output columns are present before plotting or MCDM steps.

In [ ]:
outputs


## Quickstart run: inspect generated artifacts before cleanup

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `max_items`: maximum number of items shown in logs.
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

Variables created in this cell:
- `generated_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `generated_files`.

In [ ]:
generated_files = artifact_cleaner.print_generated_files(max_items=200)

# Force-remove every generated artifact inside the dedicated result folder,
# including generated IDF files.
artifact_cleaner.delete_generated_files(
	allow_patterns=[f"{RESULTS_DIRNAME}/**"],
	deny_patterns=[],
	dry_run=True,
	remove_empty_dirs=True,
)


## Quickstart run: remove generated artifacts from this run

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
artifact_cleaner.delete_generated_files(
	allow_patterns=[f"{RESULTS_DIRNAME}/**"],
	deny_patterns=[],
	dry_run=False,
	remove_empty_dirs=True,
)
